In [ ]:
# OneLake Shortcut Auditor
> **Purpose**: Automated connectivity tests & permissions audit for OneLake shortcuts (ADLS Gen2, S3, GCS)  
> Prevents broken shortcuts and cross-workspace data leakage at scale  
> Workspace: **FabricJumpstart**

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 1 ▸ Configuration & Dependencies
# ─────────────────────────────────────────────────────────────────────────────
import requests, json, datetime, warnings
import pandas as pd
from IPython.display import display, HTML
from typing import List, Dict, Optional

warnings.filterwarnings("ignore")

# ── Workspace context (auto-populated from notebook context) ─────────────────
WORKSPACE_ID   = "f2adc10e-785a-4d71-91ec-ce4ee3aefef6"
WORKSPACE_NAME = "FabricJumpstart"

# ── Audit scope: set to None to scan ALL accessible workspaces ───────────────
AUDIT_WORKSPACE_IDS = None  # e.g., [WORKSPACE_ID, "another-ws-id"] or None for all

# ── Alert thresholds ──────────────────────────────────────────────────────────
ALERT_ON_BROKEN_SHORTCUTS     = True   # Fire alert if any shortcut fails connectivity
ALERT_ON_PERMISSION_CONFLICTS = True   # Warn if shortcut crosses sensitive workspace boundaries
TEAMS_WEBHOOK_URL             = ""     # Paste your Teams webhook URL here

print(f"✅ Config loaded")
print(f"   Current workspace : {WORKSPACE_NAME} ({WORKSPACE_ID})")
print(f"   Audit scope       : {'ALL accessible workspaces' if AUDIT_WORKSPACE_IDS is None else f'{len(AUDIT_WORKSPACE_IDS)} workspace(s)'}")
print(f"   Alert on broken   : {ALERT_ON_BROKEN_SHORTCUTS}")
print(f"   Alert on conflicts: {ALERT_ON_PERMISSION_CONFLICTS}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 2 ▸ Authentication & Fabric REST API Helpers
# ─────────────────────────────────────────────────────────────────────────────

# ── Acquire bearer token ──────────────────────────────────────────────────────
try:
    token = notebookutils.credentials.getToken('pbi')
    HEADERS = {
        "Authorization": f"Bearer {token}",
        "Content-Type":  "application/json"
    }
    print("✅ Bearer token acquired via notebookutils (user identity)")
except Exception as e:
    print(f"⚠️  notebookutils token failed: {e}")
    print("   → For service-principal pipelines, use MSAL authentication.")
    HEADERS = {}

BASE_URL = "https://api.fabric.microsoft.com/v1"
print(f"   Base URL: {BASE_URL}")

# ── Helper: Paginated GET ─────────────────────────────────────────────────────
def fabric_get_all(endpoint: str, headers: dict, params: dict = None) -> List[Dict]:
    """Paginate through Fabric REST API, collecting all results."""
    url = f"{BASE_URL}/{endpoint.lstrip('/')}"
    all_results = []
    page = 0
    max_pages = 100

    while url and page < max_pages:
        resp = requests.get(url, headers=headers, 
                           params=params if page == 0 else None, timeout=60)
        if resp.status_code in (401, 403):
            raise PermissionError(f"{resp.status_code} {resp.reason} — check token & permissions")
        resp.raise_for_status()
        
        data = resp.json()
        batch = data.get("value", [])
        all_results.extend(batch)
        
        url = data.get("continuationUri") or data.get("@odata.nextLink")
        params = None
        page += 1

    return all_results

print("✅ API helpers ready")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 3 ▸ Discover All OneLake Shortcuts
#
#  Endpoints:
#    GET /workspaces                              → list accessible workspaces
#    GET /workspaces/{id}/lakehouses              → list lakehouses in workspace
#    GET /workspaces/{id}/lakehouses/{id}/shortcuts → list shortcuts in lakehouse
# ─────────────────────────────────────────────────────────────────────────────

def discover_all_shortcuts(workspace_ids: Optional[List[str]], headers: dict) -> pd.DataFrame:
    """Enumerate all shortcuts across specified (or all) workspaces."""
    
    # ── Step 1: Get workspaces to scan ───────────────────────────────────────
    if workspace_ids is None:
        print("🔍 Enumerating ALL accessible workspaces …")
        workspaces = fabric_get_all("workspaces", headers)
    else:
        print(f"🔍 Scanning {len(workspace_ids)} specified workspace(s) …")
        workspaces = [{"id": ws_id} for ws_id in workspace_ids]
        # Fetch workspace names for display
        for ws in workspaces:
            try:
                detail = requests.get(f"{BASE_URL}/workspaces/{ws['id']}", 
                                     headers=headers, timeout=30).json()
                ws["displayName"] = detail.get("displayName", ws["id"])
            except:
                ws["displayName"] = ws["id"]

    print(f"   Found {len(workspaces)} workspace(s) to audit")

    # ── Step 2: For each workspace, get lakehouses ───────────────────────────
    all_shortcuts = []
    for ws in workspaces:
        ws_id   = ws["id"]
        ws_name = ws.get("displayName", ws_id)
        
        try:
            lakehouses = fabric_get_all(f"workspaces/{ws_id}/lakehouses", headers)
            print(f"   {ws_name}: {len(lakehouses)} lakehouse(s)")
        except Exception as e:
            print(f"   ⚠️  {ws_name}: failed to list lakehouses ({e})")
            continue

        # ── Step 3: For each lakehouse, get shortcuts ────────────────────────
        for lh in lakehouses:
            lh_id   = lh["id"]
            lh_name = lh.get("displayName", lh_id)
            
            try:
                shortcuts = fabric_get_all(
                    f"workspaces/{ws_id}/lakehouses/{lh_id}/shortcuts", headers
                )
                
                for sc in shortcuts:
                    all_shortcuts.append({
                        "workspace_id":   ws_id,
                        "workspace_name": ws_name,
                        "lakehouse_id":   lh_id,
                        "lakehouse_name": lh_name,
                        "shortcut_name":  sc.get("name", ""),
                        "shortcut_path":  sc.get("path", ""),
                        "target_type":    sc.get("target", {}).get("type", "Unknown"),
                        "target_url":     sc.get("target", {}).get("location", ""),
                        "connection_id":  sc.get("target", {}).get("connectionId", ""),
                    })
                
                if shortcuts:
                    print(f"      └─ {lh_name}: {len(shortcuts)} shortcut(s)")
                    
            except Exception as e:
                print(f"      └─ {lh_name}: failed to list shortcuts ({e})")

    df = pd.DataFrame(all_shortcuts)
    return df


# ── Execute discovery ─────────────────────────────────────────────────────────
try:
    df_shortcuts = discover_all_shortcuts(AUDIT_WORKSPACE_IDS, HEADERS)
    
    if df_shortcuts.empty:
        print("\n✅ No shortcuts found in scope.")
    else:
        print(f"\n✅ Discovery complete: {len(df_shortcuts)} shortcut(s) found")
        print("\n" + "=" * 80)
        print("  SHORTCUT INVENTORY")
        print("=" * 80)
        display(HTML(df_shortcuts.to_html(index=False, escape=False)))
        
        # ── Summary by target type ────────────────────────────────────────────
        type_summary = df_shortcuts["target_type"].value_counts()
        print(f"\n📊 By target type:\n{type_summary.to_string()}")

except Exception as e:
    print(f"❌ Shortcut discovery failed: {e}")
    df_shortcuts = pd.DataFrame()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 4 ▸ Connectivity Tests — Ping Each Shortcut Source
#
#  Tests whether each shortcut target is reachable via OneLake APIs.
#  For ADLS Gen2 / S3 / GCS shortcuts, we attempt to list the root path.
# ─────────────────────────────────────────────────────────────────────────────

def test_shortcut_connectivity(row: pd.Series, headers: dict) -> Dict:
    """
    Test if a shortcut is accessible by attempting to read its metadata.
    Returns dict with status, error_message, test_timestamp.
    """
    ws_id = row["workspace_id"]
    lh_id = row["lakehouse_id"]
    sc_path = row["shortcut_path"]
    
    # ── Construct OneLake path for the shortcut ──────────────────────────────
    # Format: /workspaces/{ws}/lakehouses/{lh}/Files/{shortcut_path}
    onelake_path = f"workspaces/{ws_id}/items/{lh_id}/Files/{sc_path.lstrip('/')}"
    test_url = f"{BASE_URL}/{onelake_path}"
    
    result = {
        "test_timestamp": datetime.datetime.utcnow().isoformat() + "Z",
        "status": "Unknown",
        "error_message": None,
    }
    
    try:
        # ── Attempt HEAD request to check existence ──────────────────────────
        resp = requests.head(test_url, headers=headers, timeout=15, allow_redirects=True)
        
        if resp.status_code == 200:
            result["status"] = "✅ OK"
        elif resp.status_code == 404:
            result["status"] = "🔴 BROKEN"
            result["error_message"] = "404 Not Found — shortcut target does not exist"
        elif resp.status_code in (401, 403):
            result["status"] = "🟡 AUTH_FAILED"
            result["error_message"] = f"{resp.status_code} {resp.reason} — insufficient permissions"
        else:
            result["status"] = "🟡 UNKNOWN_ERROR"
            result["error_message"] = f"HTTP {resp.status_code} {resp.reason}"
            
    except requests.exceptions.Timeout:
        result["status"] = "🔴 TIMEOUT"
        result["error_message"] = "Connection timeout — shortcut source unreachable"
    except requests.exceptions.ConnectionError as e:
        result["status"] = "🔴 CONNECTION_ERROR"
        result["error_message"] = f"Connection failed: {str(e)[:100]}"
    except Exception as e:
        result["status"] = "🔴 ERROR"
        result["error_message"] = str(e)[:200]

    return result


# ── Run connectivity tests ────────────────────────────────────────────────────
if not df_shortcuts.empty:
    print("🔗 Running connectivity tests on all shortcuts …\n")
    
    test_results = []
    for idx, row in df_shortcuts.iterrows():
        print(f"   [{idx+1}/{len(df_shortcuts)}] Testing {row['workspace_name']} / "
              f"{row['lakehouse_name']} / {row['shortcut_name']} …", end=" ")
        
        result = test_shortcut_connectivity(row, HEADERS)
        test_results.append(result)
        print(result["status"])

    # ── Merge results back into dataframe ─────────────────────────────────────
    df_tests = pd.DataFrame(test_results)
    df_shortcuts = pd.concat([df_shortcuts, df_tests], axis=1)

    print("\n" + "=" * 80)
    print("  CONNECTIVITY TEST RESULTS")
    print("=" * 80)
    display(HTML(df_shortcuts[["workspace_name", "lakehouse_name", "shortcut_name", 
                               "target_type", "status", "error_message"]]
                 .to_html(index=False, escape=False)))

    # ── Summary ───────────────────────────────────────────────────────────────
    status_counts = df_shortcuts["status"].value_counts()
    print(f"\n📊 Test summary:\n{status_counts.to_string()}")

    broken = df_shortcuts[df_shortcuts["status"].str.contains("BROKEN|ERROR|TIMEOUT", 
                                                              case=False, na=False)]
    if not broken.empty:
        print(f"\n🔴 {len(broken)} broken shortcut(s) detected:")
        for _, row in broken.iterrows():
            print(f"   • {row['workspace_name']} / {row['lakehouse_name']} / "
                  f"{row['shortcut_name']}: {row['error_message']}")
    else:
        print("\n✅ All shortcuts are healthy (reachable)")

else:
    print("⚠️  No shortcuts to test (discovery returned empty).")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 5 ▸ Permissions Audit — Check Workspace & Lakehouse Access
#
#  Logs who has access to each workspace/lakehouse containing shortcuts.
#  Flags potential data leakage when sensitive shortcuts cross workspace boundaries.
# ─────────────────────────────────────────────────────────────────────────────

def get_workspace_role_assignments(workspace_id: str, headers: dict) -> List[Dict]:
    """Fetch all role assignments (users/groups) for a workspace."""
    try:
        endpoint = f"workspaces/{workspace_id}/roleAssignments"
        assignments = fabric_get_all(endpoint, headers)
        return assignments
    except Exception as e:
        print(f"   ⚠️  Failed to fetch role assignments for workspace {workspace_id}: {e}")
        return []


def audit_shortcut_permissions(df_shortcuts: pd.DataFrame, headers: dict) -> pd.DataFrame:
    """
    For each shortcut, retrieve workspace role assignments and identify users/groups
    with access. Flag cross-boundary risks.
    """
    if df_shortcuts.empty:
        return pd.DataFrame()

    # ── Get unique workspaces ─────────────────────────────────────────────────
    unique_workspaces = df_shortcuts[["workspace_id", "workspace_name"]].drop_duplicates()
    
    print(f"🔐 Auditing permissions for {len(unique_workspaces)} workspace(s) …\n")
    
    ws_permissions = {}
    for _, ws_row in unique_workspaces.iterrows():
        ws_id = ws_row["workspace_id"]
        ws_name = ws_row["workspace_name"]
        
        print(f"   Workspace: {ws_name} ({ws_id})")
        assignments = get_workspace_role_assignments(ws_id, headers)
        
        if assignments:
            principals = []
            for a in assignments:
                principal = a.get("principal", {})
                principals.append({
                    "type":         principal.get("type", "Unknown"),
                    "id":           principal.get("id", ""),
                    "displayName":  principal.get("displayName", ""),
                    "role":         a.get("role", "Unknown"),
                })
            ws_permissions[ws_id] = principals
            print(f"      → {len(principals)} principal(s) with access")
        else:
            ws_permissions[ws_id] = []
            print(f"      → 0 principals (or no read permission)")

    # ── Build permissions report ──────────────────────────────────────────────
    perm_rows = []
    for _, sc_row in df_shortcuts.iterrows():
        ws_id   = sc_row["workspace_id"]
        ws_name = sc_row["workspace_name"]
        lh_name = sc_row["lakehouse_name"]
        sc_name = sc_row["shortcut_name"]
        
        principals = ws_permissions.get(ws_id, [])
        
        if principals:
            for p in principals:
                perm_rows.append({
                    "workspace_name":   ws_name,
                    "lakehouse_name":   lh_name,
                    "shortcut_name":    sc_name,
                    "principal_type":   p["type"],
                    "principal_name":   p["displayName"],
                    "principal_id":     p["id"],
                    "role":             p["role"],
                })
        else:
            # No principals found — still log the shortcut
            perm_rows.append({
                "workspace_name":   ws_name,
                "lakehouse_name":   lh_name,
                "shortcut_name":    sc_name,
                "principal_type":   "None",
                "principal_name":   "No role assignments found",
                "principal_id":     "",
                "role":             "",
            })

    df_permissions = pd.DataFrame(perm_rows)
    return df_permissions


# ── Execute permissions audit ─────────────────────────────────────────────────
if not df_shortcuts.empty:
    df_permissions = audit_shortcut_permissions(df_shortcuts, HEADERS)

    print("\n" + "=" * 80)
    print("  PERMISSIONS AUDIT — Who Has Access to Shortcuts")
    print("=" * 80)
    display(HTML(df_permissions.head(50).to_html(index=False, escape=False)))

    if len(df_permissions) > 50:
        print(f"\n   (Showing first 50 of {len(df_permissions)} total permission entries)")

    # ── Flag potential cross-workspace data leakage ───────────────────────────
    # Example heuristic: if a shortcut in "Finance" workspace is accessible by
    # users who also have access to "Marketing" workspace, flag it.
    # (This requires cross-referencing user access across workspaces; simplified here)
    
    print("\n🛡️  Cross-workspace permission analysis:")
    print("   (For production: implement custom logic to detect sensitive data crossover)")
    print("   Example: Flag if 'Viewer' role has access to shortcuts with 'Confidential' in name")
    
    # Simple demo rule: flag any "Viewer" access to shortcuts
    risky = df_permissions[
        (df_permissions["role"] == "Viewer") & 
        (df_permissions["principal_type"] != "None")
    ]
    if not risky.empty:
        print(f"\n🟡 {len(risky)} permission(s) with 'Viewer' role detected (review for least privilege):")
        viewer_summary = risky.groupby(["workspace_name", "shortcut_name"]).size().reset_index(name="viewer_count")
        display(HTML(viewer_summary.to_html(index=False, escape=False)))
    else:
        print("\n✅ No broad 'Viewer' access detected on shortcuts.")

else:
    print("⚠️  No shortcuts to audit (discovery returned empty).")
    df_permissions = pd.DataFrame()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 6 ▸ Alerting — Teams Notification (Optional)
# ─────────────────────────────────────────────────────────────────────────────

def build_audit_alert_card(df_shortcuts: pd.DataFrame, 
                           df_permissions: pd.DataFrame,
                           workspace_name: str) -> Dict:
    """Build Teams Adaptive Card payload for shortcut audit results."""
    
    broken_count = len(df_shortcuts[df_shortcuts["status"].str.contains(
        "BROKEN|ERROR|TIMEOUT", case=False, na=False)]) if "status" in df_shortcuts.columns else 0
    
    viewer_count = len(df_permissions[df_permissions["role"] == "Viewer"]) \
                   if not df_permissions.empty else 0
    
    overall_sev = "🔴 CRITICAL" if broken_count > 0 else \
                  ("🟡 WARNING" if viewer_count > 0 else "🟢 HEALTHY")
    
    body = [
        {
            "type": "TextBlock",
            "text": f"OneLake Shortcut Audit — {overall_sev}",
            "weight": "Bolder",
            "size": "Large",
            "color": "Attention" if "CRITICAL" in overall_sev else
                     ("Warning" if "WARNING" in overall_sev else "Good"),
        },
        {
            "type": "FactSet",
            "facts": [
                {"title": "Workspace", "value": workspace_name},
                {"title": "Shortcuts scanned", "value": str(len(df_shortcuts))},
                {"title": "Broken shortcuts", "value": str(broken_count)},
                {"title": "Viewer permissions", "value": str(viewer_count)},
                {"title": "Timestamp (UTC)", "value": datetime.datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")},
            ],
        },
    ]
    
    if broken_count > 0:
        body.append({
            "type": "TextBlock",
            "text": f"🔴 {broken_count} shortcut(s) are broken or unreachable. "
                    "Review connectivity and target credentials.",
            "wrap": True,
            "color": "Attention",
        })
    
    if viewer_count > 0:
        body.append({
            "type": "TextBlock",
            "text": f"🟡 {viewer_count} 'Viewer' role assignment(s) detected. "
                    "Review for least-privilege access.",
            "wrap": True,
            "color": "Warning",
        })

    return {
        "type": "message",
        "attachments": [{
            "contentType": "application/vnd.microsoft.card.adaptive",
            "content": {
                "$schema": "http://adaptivecards.io/schemas/adaptive-card.json",
                "type": "AdaptiveCard",
                "version": "1.4",
                "body": body,
                "actions": [{
                    "type": "Action.OpenUrl",
                    "title": "Open Workspace in Fabric",
                    "url": f"https://app.fabric.microsoft.com/groups/{WORKSPACE_ID}",
                }],
            },
        }],
    }


# ── Evaluate alert conditions ─────────────────────────────────────────────────
should_alert = False
alert_reasons = []

if not df_shortcuts.empty and "status" in df_shortcuts.columns:
    broken = df_shortcuts[df_shortcuts["status"].str.contains(
        "BROKEN|ERROR|TIMEOUT", case=False, na=False)]
    if not broken.empty and ALERT_ON_BROKEN_SHORTCUTS:
        should_alert = True
        alert_reasons.append(f"{len(broken)} broken shortcut(s)")

if not df_permissions.empty:
    viewers = df_permissions[df_permissions["role"] == "Viewer"]
    if not viewers.empty and ALERT_ON_PERMISSION_CONFLICTS:
        should_alert = True
        alert_reasons.append(f"{len(viewers)} 'Viewer' permission(s)")

# ── Send alert if conditions met ──────────────────────────────────────────────
if should_alert:
    print(f"🚨 Alert triggered: {', '.join(alert_reasons)}")
    
    card = build_audit_alert_card(df_shortcuts, df_permissions, WORKSPACE_NAME)
    print("\n📨 Adaptive Card payload (preview):")
    print(json.dumps(card, indent=2)[:1000], "…\n")
    
    if TEAMS_WEBHOOK_URL:
        try:
            r = requests.post(TEAMS_WEBHOOK_URL, json=card, timeout=15)
            r.raise_for_status()
            print(f"✅ Teams alert sent (HTTP {r.status_code})")
        except Exception as ex:
            print(f"❌ Teams delivery failed: {ex}")
    else:
        print("⚠️  TEAMS_WEBHOOK_URL not set in Cell 1 — alert payload ready but not sent.")
else:
    print("✅ No alert conditions met — all shortcuts healthy and permissions appropriate.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 7 ▸ Summary & Optional Lakehouse Persistence
# ─────────────────────────────────────────────────────────────────────────────

run_summary = {
    "run_timestamp_utc":    datetime.datetime.utcnow().isoformat() + "Z",
    "workspace_name":       WORKSPACE_NAME,
    "shortcuts_scanned":    len(df_shortcuts) if not df_shortcuts.empty else 0,
    "broken_shortcuts":     len(df_shortcuts[df_shortcuts["status"].str.contains(
                                "BROKEN|ERROR|TIMEOUT", case=False, na=False)]) 
                                if not df_shortcuts.empty and "status" in df_shortcuts.columns else 0,
    "viewer_permissions":   len(df_permissions[df_permissions["role"] == "Viewer"]) 
                                if not df_permissions.empty else 0,
    "alert_triggered":      should_alert,
}

print("=" * 80)
print("  ONELAKE SHORTCUT AUDIT — EXECUTIVE SUMMARY")
print("=" * 80)
for k, v in run_summary.items():
    print(f"  {k:<30} {v}")
print("=" * 80)

# ── Optional: persist to Lakehouse Delta tables for historical trending ───────
# Uncomment after attaching a Lakehouse to this notebook
#
# LAKEHOUSE_SHORTCUTS_TABLE = (
#     "abfss://<workspace>@onelake.dfs.fabric.microsoft.com"
#     "/<lakehouse>.Lakehouse/Tables/shortcut_audit_results"
# )
# LAKEHOUSE_PERMISSIONS_TABLE = (
#     "abfss://<workspace>@onelake.dfs.fabric.microsoft.com"
#     "/<lakehouse>.Lakehouse/Tables/shortcut_permissions"
# )
#
# if not df_shortcuts.empty:
#     df_shortcuts["audit_run_timestamp"] = run_summary["run_timestamp_utc"]
#     spark_df = spark.createDataFrame(df_shortcuts)
#     spark_df.write.format("delta").mode("append").save(LAKEHOUSE_SHORTCUTS_TABLE)
#     print("✅ Shortcuts audit results written to Lakehouse Delta table")
#
# if not df_permissions.empty:
#     df_permissions["audit_run_timestamp"] = run_summary["run_timestamp_utc"]
#     spark_df_perm = spark.createDataFrame(df_permissions)
#     spark_df_perm.write.format("delta").mode("append").save(LAKEHOUSE_PERMISSIONS_TABLE)
#     print("✅ Permissions audit results written to Lakehouse Delta table")

print("\n📌 Next Steps:")
print("  1. Set AUDIT_WORKSPACE_IDS in Cell 1 to target specific workspaces (or None for all).")
print("  2. Set TEAMS_WEBHOOK_URL in Cell 1 for automated alerts.")
print("  3. Review broken shortcuts in Cell 4 output and fix connectivity or credentials.")
print("  4. Review 'Viewer' permissions in Cell 5 and apply least-privilege access.")
print("  5. Schedule via Fabric Pipeline (cron: daily) for ongoing monitoring.")
print("  6. Uncomment Lakehouse persistence in Cell 7 to build trending reports in Power BI.")
print("  7. Customize cross-workspace detection logic in Cell 5 for your org's data classification.")